# Notebook 1: Read & Join the Tables
**MLOps Training 2026/2027 · Task 2: From Tables to Notebooks**

### Objective:
1. Connect to the local PostgreSQL `olist` database using credentials loaded from `.env`.
2. Ingest and inspect all 9 relational tables individually on their own (understand schemas, row counts, keys, duplicates, and grain).
3. Aggregate multi-row tables (`order_items`, `order_payments`, `geolocation`) to **1 row per order** before joining to prevent row duplication.
4. Join the tables into a single master ML table where each row represents a unique order (`order_id`).
5. Perform integrity and sanity checks (confirm 1 row per order, check missing values).
6. Save the resulting table as **Parquet** and **CSV** artifacts for Notebook 2 (`Create Labels`).

In [1]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from IPython.display import display

# Display configuration
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Locate repository root and artifacts directory
def find_project_root(start_path: Path) -> Path:
    """Traverse up directory tree to locate repository root containing .env or .git."""
    for parent in [start_path] + list(start_path.parents):
        if (parent / ".env").exists() or (parent / ".git").exists():
            return parent
    return start_path

root_dir = find_project_root(Path.cwd())
load_dotenv(dotenv_path=root_dir / ".env")

# Define artifact output directory for Notebook 1
artifact_dir = root_dir / "tasks" / "task-02-notebooks" / "artifacts" / "01_read_and_join"
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {root_dir}")
print(f"Artifact directory: {artifact_dir}")

Project root: /Users/halakhalifa/Desktop/qafza/olist-mlops
Artifact directory: /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/01_read_and_join


In [3]:
# Database connection settings from .env
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist")
DB_USER = os.getenv("DB_USER") or os.getenv("USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

# Build PostgreSQL connection URI
if DB_PASSWORD:
    db_uri = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
else:
    db_uri = f"postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(db_uri)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, version();")).fetchone()
    print(f"Connected to Database : {result[0]}")
    print(f"Database User        : {result[1]}")
    print(f"PostgreSQL Version   : {result[2][:35]}...")

Connected to Database : olist
Database User        : halakhalifa
PostgreSQL Version   : PostgreSQL 14.23 (Homebrew) on aarc...


## Step 1: Ingest All 9 Tables from PostgreSQL

We read each table from PostgreSQL into a dedicated pandas DataFrame in memory.

In [4]:
# List of all 9 tables in the Olist dataset
table_names = [
    "customers",
    "orders",
    "order_items",
    "products",
    "sellers",
    "order_payments",
    "order_reviews",
    "geolocation",
    "product_category_translation"
]

tables = {}
for tbl in table_names:
    query = f"SELECT * FROM {tbl};"
    df_tbl = pd.read_sql(query, engine)
    tables[tbl] = df_tbl
    print(f"Loaded '{tbl:<28}': {df_tbl.shape[0]:>8,} rows, {df_tbl.shape[1]:>2} columns")

Loaded 'customers                   ':   99,441 rows,  5 columns


Loaded 'orders                      ':   99,441 rows,  8 columns


Loaded 'order_items                 ':  112,650 rows,  7 columns


Loaded 'products                    ':   32,951 rows,  9 columns
Loaded 'sellers                     ':    3,095 rows,  4 columns


Loaded 'order_payments              ':  103,886 rows,  5 columns


Loaded 'order_reviews               ':   99,224 rows,  7 columns


Loaded 'geolocation                 ': 1,000,163 rows,  5 columns
Loaded 'product_category_translation':       71 rows,  2 columns


## Step 2: Inspect Each Table on Its Own

Rather than assuming schemas and relationships, we inspect each table individually by:
1. Previewing actual sample rows (`head`).
2. Checking dimensions, data types, and missing values (`isna`).
3. Testing candidate primary keys for uniqueness (`is_unique`, `nunique`).
4. Determining **what 1 row means (the grain)** and identifying any join fan-out / row duplication risks.

### 2.1 `orders` Table
The central backbone table representing the lifecycle and timestamps of customer orders.

In [5]:
df_orders = tables["orders"]
print(f"Shape: {df_orders.shape[0]:,} rows x {df_orders.shape[1]} columns")
print(f"Is order_id unique? {df_orders['order_id'].is_unique} ({df_orders['order_id'].nunique():,} unique values)")
print(f"Full duplicate rows: {df_orders.duplicated().sum()}")
display(df_orders.head(3))

# Grain discovery:
print("-> Grain: Exactly 1 row = 1 unique order. This will serve as our Master ML base table.")

Shape: 99,441 rows x 8 columns
Is order_id unique? True (99,441 unique values)
Full duplicate rows: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


-> Grain: Exactly 1 row = 1 unique order. This will serve as our Master ML base table.


### 2.2 `customers` Table
Contains customer location information. Notice the distinction between `customer_id` and `customer_unique_id`.

In [6]:
df_customers = tables["customers"]
print(f"Shape: {df_customers.shape[0]:,} rows x {df_customers.shape[1]} columns")
print(f"Unique customer_id: {df_customers['customer_id'].nunique():,} (Unique: {df_customers['customer_id'].is_unique})")
print(f"Unique customer_unique_id: {df_customers['customer_unique_id'].nunique():,} (Tracks repeat buyers across multiple orders)")
display(df_customers.head(3))

# Grain discovery:
print("-> Grain: 1 row = 1 customer order session. Joins 1:1 with orders.customer_id.")

Shape: 99,441 rows x 5 columns


Unique customer_id: 99,441 (Unique: True)
Unique customer_unique_id: 96,096 (Tracks repeat buyers across multiple orders)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


-> Grain: 1 row = 1 customer order session. Joins 1:1 with orders.customer_id.


### 2.3 `order_items` Table
Contains product items purchased within each order. Let's verify if `order_id` is unique or if orders have multiple items.

In [7]:
df_items = tables["order_items"]
print(f"Shape: {df_items.shape[0]:,} rows x {df_items.shape[1]} columns")
print(f"Is order_id unique? {df_items['order_id'].is_unique} ({df_items['order_id'].nunique():,} distinct orders across {len(df_items):,} item rows)")
print(f"Composite key (order_id, order_item_id) unique? {df_items.set_index(['order_id', 'order_item_id']).index.is_unique}")

# View an order with multiple items:
multi_item_order = df_items[df_items["order_id"].duplicated(keep=False)]["order_id"].iloc[0]
print(f"\nExample: Multi-item order ({multi_item_order}):")
display(df_items[df_items["order_id"] == multi_item_order])

# Grain discovery & Action required:
print("-> Grain: 1 row = 1 purchased item in an order.")
print("-> WARNING: A direct join on order_id would multiply rows (fan-out). We MUST aggregate order_items by order_id before joining!")

Shape: 112,650 rows x 7 columns
Is order_id unique? False (98,666 distinct orders across 112,650 item rows)


Composite key (order_id, order_item_id) unique? True

Example: Multi-item order (0008288aa423d2a3f00fcb17cd7d8719):


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
13,0008288aa423d2a3f00fcb17cd7d8719,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.9,13.37
14,0008288aa423d2a3f00fcb17cd7d8719,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.9,13.37


-> Grain: 1 row = 1 purchased item in an order.
-> WARNING: A direct join on order_id would multiply rows (fan-out). We MUST aggregate order_items by order_id before joining!


### 2.4 `order_payments` Table
Records payment methods, installments, and split transactions.

In [8]:
df_payments = tables["order_payments"]
print(f"Shape: {df_payments.shape[0]:,} rows x {df_payments.shape[1]} columns")
print(f"Is order_id unique? {df_payments['order_id'].is_unique} ({df_payments['order_id'].nunique():,} distinct orders across {len(df_payments):,} rows)")
print(f"Composite key (order_id, payment_sequential) unique? {df_payments.set_index(['order_id', 'payment_sequential']).index.is_unique}")

# Example of an order with split payment methods (e.g. credit card + voucher):
multi_pay_order = df_payments[df_payments["order_id"].duplicated(keep=False)]["order_id"].iloc[0]
print(f"\nExample: Multi-payment order ({multi_pay_order}):")
display(df_payments[df_payments["order_id"] == multi_pay_order])

# Grain discovery & Action required:
print("-> Grain: 1 row = 1 payment split/installment for an order.")
print("-> WARNING: A direct join on order_id would multiply rows. We MUST aggregate order_payments by order_id before joining!")

Shape: 103,886 rows x 5 columns
Is order_id unique? False (99,440 distinct orders across 103,886 rows)


Composite key (order_id, payment_sequential) unique? True

Example: Multi-payment order (5cfd514482e22bc992e7693f0e3e8df7):


,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,4,665.41


-> Grain: 1 row = 1 payment split/installment for an order.
-> WARNING: A direct join on order_id would multiply rows. We MUST aggregate order_payments by order_id before joining!


### 2.5 `products` & `product_category_translation` Tables
Contains catalog specs (weights, dimensions) and Portuguese-to-English category translations.

In [9]:
df_products = tables["products"]
df_trans = tables["product_category_translation"]
print(f"Products shape: {df_products.shape[0]:,} rows x {df_products.shape[1]} columns. Unique product_id: {df_products['product_id'].is_unique}")
print(f"Category translations shape: {df_trans.shape[0]} rows x {df_trans.shape[1]} columns. Unique Portuguese name: {df_trans['product_category_name'].is_unique}")
display(df_products.head(3))
display(df_trans.head(3))

# Grain discovery:
print("-> Products Grain: 1 row = 1 catalog product.")
print("-> Translation Grain: 1 row = 1 category translation mapping.")

Products shape: 32,951 rows x 9 columns. Unique product_id: True
Category translations shape: 71 rows x 2 columns. Unique Portuguese name: True


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


-> Products Grain: 1 row = 1 catalog product.
-> Translation Grain: 1 row = 1 category translation mapping.


### 2.6 `sellers` Table
Stores registered marketplace sellers and their zip codes.

In [10]:
df_sellers = tables["sellers"]
print(f"Shape: {df_sellers.shape[0]:,} rows x {df_sellers.shape[1]} columns")
print(f"Is seller_id unique? {df_sellers['seller_id'].is_unique}")
display(df_sellers.head(3))

# Grain discovery:
print("-> Grain: 1 row = 1 registered marketplace seller.")

Shape: 3,095 rows x 4 columns
Is seller_id unique? True


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


-> Grain: 1 row = 1 registered marketplace seller.


### 2.7 `order_reviews` Table
Stores customer survey feedback, score (1 to 5), and review timestamps.

> **Target Leakage Note**: Customer reviews are submitted **after** an order is delivered. Using post-delivery review data to predict pre-delivery late status would introduce target leakage. Therefore, `order_reviews` is intentionally excluded from the master ML feature table.

In [11]:
df_reviews = tables["order_reviews"]
print(f"Shape: {df_reviews.shape[0]:,} rows x {df_reviews.shape[1]} columns")
print(f"Unique review_id: {df_reviews['review_id'].nunique():,}, Unique order_id: {df_reviews['order_id'].nunique():,}")
display(df_reviews.head(3))

# Grain discovery:
print("-> Grain: 1 row = 1 customer review submission for an order.")

Shape: 99,224 rows x 7 columns
Unique review_id: 98,410, Unique order_id: 98,673


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24


-> Grain: 1 row = 1 customer review submission for an order.


### 2.8 `geolocation` Table
Contains GPS coordinate readings for Brazilian postal code prefixes (`geolocation_zip_code_prefix`). Let's observe why it contains over 1 million rows for ~19k zip codes.

In [12]:
df_geo = tables["geolocation"]
print(f"Shape: {df_geo.shape[0]:,} rows x {df_geo.shape[1]} columns")
print(f"Unique zip code prefixes: {df_geo['geolocation_zip_code_prefix'].nunique():,}")
print(f"Average coordinate readings per zip prefix: {len(df_geo) / df_geo['geolocation_zip_code_prefix'].nunique():.1f}")

# Preview multiple coordinates recorded for a single zip code prefix:
sample_zip = df_geo["geolocation_zip_code_prefix"].iloc[0]
print(f"\nSample zip prefix ({sample_zip}) coordinates:")
display(df_geo[df_geo["geolocation_zip_code_prefix"] == sample_zip].head(4))

# Grain discovery & Warning:
print("-> Grain: 1 row = 1 recorded GPS reading for a postal code prefix.")
print("-> CRITICAL WARNING: A direct join on zip_code_prefix would cause massive row explosion (fan-out into millions of rows)!")
print("-> We MUST aggregate coordinates (mean/median lat/lng) per zip_code_prefix before joining.")

Shape: 1,000,163 rows x 5 columns
Unique zip code prefixes: 19,015
Average coordinate readings per zip prefix: 52.6

Sample zip prefix (1037) coordinates:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
14,1037,-23.545187,-46.637855,são paulo,SP
31,1037,-23.546705,-46.640336,são paulo,SP
169,1037,-23.543883,-46.638075,são paulo,SP


-> Grain: 1 row = 1 recorded GPS reading for a postal code prefix.
-> CRITICAL WARNING: A direct join on zip_code_prefix would cause massive row explosion (fan-out into millions of rows)!
-> We MUST aggregate coordinates (mean/median lat/lng) per zip_code_prefix before joining.


## Step 3: Pre-Join Aggregations (Grain Alignment to 1 Row Per Order)

From our individual table inspections in Step 2, we identified the three tables that require aggregation before joining to prevent row duplication:

1. **`products` + `product_category_translation`**:
   - Merge English category names and calculate physical product volume in cm³ (`length * height * width`).

2. **`order_items` &rarr; `items_agg`** (1 row per order):
   - `item_count`, `total_price`, `total_freight`, `avg_item_price`, `avg_item_freight`, `total_weight_g`, `total_volume_cm3`, `num_sellers`, `primary_seller_id`, and `primary_product_category`.

3. **`order_payments` &rarr; `payments_agg`** (1 row per order):
   - `total_payment_value`, `payment_installments_max`, `payment_transactions_count`, `dominant_payment_type`.

4. **`geolocation` &rarr; `geo_agg`** (1 row per zip prefix):
   - Aggregate GPS points by taking the median `lat` and `lng` per postal code prefix.

In [13]:
# 1. Merge products with English category translations
df_products = tables["products"].copy()
df_trans = tables["product_category_translation"].copy()

df_products = df_products.merge(
    df_trans, 
    on="product_category_name", 
    how="left"
)

# Fill unmapped English category names with the original category name, or unknown
df_products["product_category_name_english"] = (
    df_products["product_category_name_english"]
    .fillna(df_products["product_category_name"])
    .fillna("unknown")
)

# Calculate product volume in cm3 (length * height * width)
df_products["product_volume_cm3"] = (
    df_products["product_length_cm"] * 
    df_products["product_height_cm"] * 
    df_products["product_width_cm"]
)

print(f"Enriched products table shape: {df_products.shape}")
display(df_products[["product_id", "product_category_name_english", "product_weight_g", "product_volume_cm3"]].head())

Enriched products table shape: (32951, 11)


,product_id,product_category_name_english,product_weight_g,product_volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,225.0,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1000.0,10800.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,154.0,2430.0
3,cef67bcfe19066a932b7673e239eb23d,baby,371.0,2704.0
4,9dc1a7de274444849c219cff195d0b71,housewares,625.0,4420.0


In [14]:
# Merge order_items with enriched products and sellers to capture item-level dimensions
items_enriched = tables["order_items"].merge(
    df_products[["product_id", "product_category_name_english", "product_weight_g", "product_volume_cm3"]],
    on="product_id",
    how="left"
).merge(
    tables["sellers"][["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

# Aggregate order_items to 1 row per order_id
# For multi-item orders with multiple sellers/categories, we determine the primary seller and primary category (highest priced item)
items_sorted = items_enriched.sort_values(["order_id", "price"], ascending=[True, False])
primary_items = items_sorted.drop_duplicates(subset=["order_id"]).copy()

agg_dict = {
    "order_item_id": "count",
    "price": ["sum", "mean"],
    "freight_value": ["sum", "mean"],
    "product_weight_g": "sum",
    "product_volume_cm3": "sum",
    "seller_id": "nunique"
}

items_grouped = items_enriched.groupby("order_id").agg(agg_dict)
items_grouped.columns = [
    "item_count",
    "total_price",
    "avg_item_price",
    "total_freight",
    "avg_item_freight",
    "total_weight_g",
    "total_volume_cm3",
    "num_sellers"
]
items_grouped = items_grouped.reset_index()

# Merge primary seller and product category info
items_agg = items_grouped.merge(
    primary_items[[
        "order_id", 
        "seller_id", 
        "seller_zip_code_prefix", 
        "seller_city", 
        "seller_state",
        "product_category_name_english"
    ]].rename(columns={
        "seller_id": "primary_seller_id",
        "seller_zip_code_prefix": "primary_seller_zip_code",
        "seller_city": "primary_seller_city",
        "seller_state": "primary_seller_state",
        "product_category_name_english": "primary_product_category"
    }),
    on="order_id",
    how="left"
)

print(f"Aggregated order items: {items_agg.shape[0]:,} unique orders.")
display(items_agg.head())

Aggregated order items: 98,666 unique orders.


,order_id,item_count,total_price,avg_item_price,total_freight,avg_item_freight,total_weight_g,total_volume_cm3,num_sellers,primary_seller_id,primary_seller_zip_code,primary_seller_city,primary_seller_state,primary_product_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,58.90,13.29,13.29,650.0,3528.0,1,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,239.90,19.93,19.93,30000.0,60000.0,1,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,199.00,199.00,17.87,17.87,3050.0,14157.0,1,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.99,12.79,12.79,200.0,2400.0,1,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,199.90,18.14,18.14,3750.0,42000.0,1,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,garden_tools


In [15]:
# Aggregate order_payments to 1 row per order_id
df_pay = tables["order_payments"]

# 1. Numerical aggregations
pay_numeric = df_pay.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "max",
    "payment_sequential": "count"
}).reset_index().rename(columns={
    "payment_value": "total_payment_value",
    "payment_installments": "payment_installments_max",
    "payment_sequential": "payment_transactions_count"
})

# 2. Find dominant payment type per order (payment method with largest value)
dominant_pay = df_pay.sort_values(["order_id", "payment_value"], ascending=[True, False]).drop_duplicates(subset=["order_id"])[["order_id", "payment_type"]]
dominant_pay.columns = ["order_id", "dominant_payment_type"]

# Merge payment features
payments_agg = pay_numeric.merge(dominant_pay, on="order_id", how="left")

print(f"Aggregated payments: {payments_agg.shape[0]:,} unique orders.")
display(payments_agg.head())

Aggregated payments: 99,440 unique orders.


,order_id,total_payment_value,payment_installments_max,payment_transactions_count,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,credit_card


In [16]:
# Aggregate geolocation to 1 unique row per zip_code_prefix (median latitude and longitude)
geo_agg = tables["geolocation"].groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "median",
    "geolocation_lng": "median"
}).reset_index().rename(columns={
    "geolocation_lat": "geo_lat",
    "geolocation_lng": "geo_lng"
})

print(f"Aggregated geolocation: {geo_agg.shape[0]:,} unique zip code prefixes.")
display(geo_agg.head())

Aggregated geolocation: 19,015 unique zip code prefixes.


,geolocation_zip_code_prefix,geo_lat,geo_lng
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


## Step 4: Join into Single Master ML Table (1 Row Per Order)

We join all tables to `orders` using `left` joins:
1. `orders` (base table, 1 row per order)
2. `customers` (joined on `customer_id`)
3. `items_agg` (joined on `order_id`)
4. `payments_agg` (joined on `order_id`)
5. `geo_agg` (joined for customer location on `customer_zip_code_prefix`)
6. `geo_agg` (joined for seller location on `primary_seller_zip_code`)

In [17]:
df_orders = tables["orders"].copy()
df_customers = tables["customers"].copy()

# 1. Merge orders with customers
ml_table = df_orders.merge(
    df_customers, 
    on="customer_id", 
    how="left"
)

# 2. Merge with aggregated items
ml_table = ml_table.merge(
    items_agg, 
    on="order_id", 
    how="left"
)

# 3. Merge with aggregated payments
ml_table = ml_table.merge(
    payments_agg, 
    on="order_id", 
    how="left"
)

# 4. Merge customer geolocation
ml_table = ml_table.merge(
    geo_agg.rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "geo_lat": "customer_lat",
        "geo_lng": "customer_lng"
    }),
    on="customer_zip_code_prefix",
    how="left"
)

# 5. Merge seller geolocation
ml_table = ml_table.merge(
    geo_agg.rename(columns={
        "geolocation_zip_code_prefix": "primary_seller_zip_code",
        "geo_lat": "seller_lat",
        "geo_lng": "seller_lng"
    }),
    on="primary_seller_zip_code",
    how="left"
)

print(f"Master ML Table Shape: {ml_table.shape} (Rows: {ml_table.shape[0]:,}, Columns: {ml_table.shape[1]})")

Master ML Table Shape: (99441, 33) (Rows: 99,441, Columns: 33)


## Step 5: Data Integrity & Validation Checks

We verify that:
1. Row count matches the total number of orders in `orders` (99,441).
2. Zero duplicated `order_id`s exist.
3. Key timestamps are present for label creation in Notebook 2.

In [18]:
# 1. Verify row count
assert len(ml_table) == len(df_orders), f"Row count mismatch! Expected {len(df_orders)}, got {len(ml_table)}"
print(f"✓ Row count check passed: Exactly {len(ml_table):,} rows.")

# 2. Verify uniqueness of order_id
assert ml_table["order_id"].is_unique, "Duplicate order_id found!"
print(f"✓ Uniqueness check passed: Every row represents a unique order.")

# 3. Check order status distribution
print("\nOrder Status Distribution:")
print(ml_table["order_status"].value_counts())

# 4. Check date column null counts
date_cols = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]
print("\nDate Columns Null Counts:")
print(ml_table[date_cols].isna().sum())

# 5. Preview first 3 rows
display(ml_table.head(3).T)

✓ Row count check passed: Exactly 99,441 rows.
✓ Uniqueness check passed: Every row represents a unique order.

Order Status Distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Date Columns Null Counts:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


,0,1,2
order_id,e481f51cbdc54678b7cc49136f2d6af7,53cdb2fc8bc7dce0b6741e2150273451,47770eb9100c2d0c44946d9cf07ec65d
customer_id,9ef432eb6251297304e76186b10a928d,b0830fb4747a6c6d20dea0b8c802d7ef,41ce2a54c0b03bf3443c3d931a367089
order_status,delivered,delivered,delivered
order_purchase_timestamp,2017-10-02 10:56:33,2018-07-24 20:41:37,2018-08-08 08:38:49
order_approved_at,2017-10-02 11:07:15,2018-07-26 03:24:27,2018-08-08 08:55:23
order_delivered_carrier_date,2017-10-04 19:55:00,2018-07-26 14:31:00,2018-08-08 13:50:00
order_delivered_customer_date,2017-10-10 21:25:13,2018-08-07 15:27:45,2018-08-17 18:06:29
order_estimated_delivery_date,2017-10-18 00:00:00,2018-08-13 00:00:00,2018-09-04 00:00:00
customer_unique_id,7c396fd4830fd04220f754e42b4e5bff,af07308b275d755c9edb36a90c618231,3a653a41f6f9fc3d2a113cf8398680e8
customer_zip_code_prefix,3149,47813,75265


## Step 6: Save Artifacts for Notebook 2

We save the master table to `tasks/task-02-notebooks/artifacts/01_read_and_join/` in both **Parquet** and **CSV** formats.

In [19]:
parquet_path = artifact_dir / "joined_orders.parquet"
csv_path = artifact_dir / "joined_orders.csv"

# Save Parquet (efficient, preserves dtypes and schema)
ml_table.to_parquet(parquet_path, index=False)
print(f"✓ Saved Parquet artifact: {parquet_path} ({parquet_path.stat().st_size / (1024*1024):.2f} MB)")

# Save CSV
ml_table.to_csv(csv_path, index=False)
print(f"✓ Saved CSV artifact: {csv_path} ({csv_path.stat().st_size / (1024*1024):.2f} MB)")

print("\n=======================================================")
print("✓ Notebook 1 execution completed successfully!")
print("✓ Artifacts are ready for Notebook 2 (Create Labels).")
print("=======================================================")

✓ Saved Parquet artifact: /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/01_read_and_join/joined_orders.parquet (16.71 MB)


✓ Saved CSV artifact: /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/01_read_and_join/joined_orders.csv (41.20 MB)

✓ Notebook 1 execution completed successfully!
✓ Artifacts are ready for Notebook 2 (Create Labels).
